In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/cleaned/walmart_sales_cleaned.csv", parse_dates=["Date"])

In [2]:
# 1. Calendar features
df["Week"]       = df["Date"].dt.isocalendar().week
df["Month"]      = df["Date"].dt.month
df["Quarter"]    = df["Date"].dt.quarter
df["Year"]       = df["Date"].dt.year
df["Month_sin"]  = np.sin(2 * np.pi * df["Month"]/12)
df["Month_cos"]  = np.cos(2 * np.pi * df["Month"]/12)

In [3]:
# 2. Holiday flags (expand if you want more granularity)
df["IsNearChristmas"] = ((df["Month"] == 12) | ((df["Month"] == 1) & (df["Week"] <= 2))).astype(int)

In [4]:
# 3. Lags (group by store + dept!)
df = df.sort_values(["Store", "Dept", "Date"])

df["Lag1"]  = df.groupby(["Store", "Dept"])["Weekly_Sales"].shift(1)
df["Lag4"]  = df.groupby(["Store", "Dept"])["Weekly_Sales"].shift(4)
df["Lag52"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].shift(52)   # same week last year

In [5]:
# 4. Rolling (also grouped!)
df["RollingMean4"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].rolling(4, min_periods=2).mean().reset_index(level=[0,1], drop=True)

In [6]:
# 5. More lag & lag differences (momentum / recent change)
df["Lag2"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].shift(2)
df["Lag3"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].shift(3)

df["Sales_Change_1w"] = df["Weekly_Sales"] - df["Lag1"]           # raw difference
df["Sales_Pct_Change_1w"] = (df["Weekly_Sales"] - df["Lag1"]) / df["Lag1"].replace(0, np.nan)
df["Sales_Pct_Change_1w"] = df["Sales_Pct_Change_1w"].clip(-5, 5)  # prevent crazy outliers

In [7]:
# 6. Rolling statistics — more windows
df["RollingStd4"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].rolling(4, min_periods=2).std().reset_index(level=[0,1], drop=True)
df["RollingMean12"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].rolling(12, min_periods=4).mean().reset_index(level=[0,1], drop=True)

In [9]:
# 7. Target encoding / historical averages (very powerful for store & dept)
# Average sales per store-dept combination (full history)
df["Store_Dept_Avg"] = df.groupby(["Store", "Dept"])["Weekly_Sales"].transform("mean")

# Average sales per store (across all depts)
df["Store_Avg"] = df.groupby("Store")["Weekly_Sales"].transform("mean")

# Average sales per department (across all stores)
df["Dept_Avg"] = df.groupby("Dept")["Weekly_Sales"].transform("mean")

# Relative strength: how much above/below usual for this store-dept
df["Sales_Relative_To_StoreDept"] = df["Weekly_Sales"] / df["Store_Dept_Avg"].replace(0, np.nan)

In [17]:
# 8. External variable transformations / interactions
# Temperature deviation from 65°F (common comfort point)
df["Temp_Deviation"] = df["Temperature"] - 65

# Fuel price change last week (price shock)
df["Fuel_Change_1w"] = df.groupby(["Store"])["Fuel_Price"].diff(1)

# Markdown intensity (relative to sales)
df["Markdown_Ratio"] = df["MarkdownTotal"] / df["Weekly_Sales"].clip(1)   # avoid div-by-zero

In [11]:
# 9. Cyclical encoding for week of year (better than raw week number)
df["Week_sin"] = np.sin(2 * np.pi * df["Week"] / 52)
df["Week_cos"] = np.cos(2 * np.pi * df["Week"] / 52)

In [16]:
# 10. Simple promotion flag (many people find this helpful)
df["Has_Any_Markdown"] = (df["MarkdownTotal"] > 0).astype(int)
df["High_Markdown"] = (df["MarkdownTotal"] > df["MarkdownTotal"].quantile(0.75)).astype(int)

In [13]:
# 11. Black Friday / Thanksgiving approximate flag
# Rough heuristic — week 47–48 of each year is often BF week
df["Is_BlackFriday_Week"] = ((df["Week"] >= 47) & (df["Week"] <= 48)).astype(int)

In [19]:
#Quick check before saving
print("Columns now:", df.columns.tolist())
print("\nMissing % after all features:")
print(df.isna().mean().sort_values(ascending=False).head(12))

Columns now: ['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday_x', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'IsHoliday_y', 'Type', 'Size', 'Week', 'Month', 'Quarter', 'Year', 'Month_sin', 'Month_cos', 'IsNearChristmas', 'Lag1', 'Lag4', 'Lag52', 'RollingMean4', 'Lag2', 'Lag3', 'Sales_Change_1w', 'Sales_Pct_Change_1w', 'RollingStd4', 'RollingMean12', 'Store_Dept_Avg', 'Store_Avg', 'Dept_Avg', 'Sales_Relative_To_StoreDept', 'Temp_Deviation', 'Fuel_Change_1w', 'Week_sin', 'Week_cos', 'Is_BlackFriday_Week', 'MarkdownTotal', 'Has_Any_Markdown', 'Markdown_Ratio', 'High_Markdown']

Missing % after all features:
MarkDown2         0.736110
MarkDown4         0.679847
MarkDown3         0.674808
MarkDown1         0.642572
MarkDown5         0.640790
MarkdownTotal     0.640790
Markdown_Ratio    0.640790
Lag52             0.380689
Lag4              0.031155
RollingMean12     0.023458
Lag3              0.023458
Lag2         

In [20]:
# ─── Imputation before modeling ───────────────────────────────────────

# Markdowns: missing = no promotion → 0
markdown_cols = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5','MarkdownTotal','Markdown_Ratio']
df[markdown_cols] = df[markdown_cols].fillna(0)

# High_Markdown depends on MarkdownTotal → recalculate after fill
df['High_Markdown'] = (df['MarkdownTotal'] > df['MarkdownTotal'].quantile(0.75)).astype(int)

# Lags & rollings: fill with group average (store-dept specific)
group_cols = ['Store', 'Dept']

for col in ['Lag1','Lag2','Lag3','Lag4','Lag52',
            'RollingMean4','RollingStd4','RollingMean12',
            'Sales_Change_1w','Sales_Pct_Change_1w']:
    df[col] = df.groupby(group_cols)[col].transform(lambda x: x.fillna(x.mean()))

# If any remaining NaN (very first rows), forward fill or global mean as last resort
df = df.fillna(df.mean(numeric_only=True))

# Final check
print("\nMissing values AFTER imputation:")
print(df.isna().sum().sum(), "remaining missing values")

print("\nFinal shape:", df.shape)


Missing values AFTER imputation:
0 remaining missing values

Final shape: (421570, 47)


In [22]:
# Save two versions — one raw, one imputed
df.to_csv("../data/processed/walmart_features_raw.csv", index=False)
print("Saved raw version with NaNs: ../data/processed/walmart_features_raw.csv")

# Imputed version (most models prefer this)
df_imputed = df.copy()  # already imputed above
df_imputed.to_csv("../data/processed/walmart_features_imputed.csv", index=False)
print("Saved imputed version: ../data/processed/walmart_features_imputed.csv")

Saved raw version with NaNs: ../data/processed/walmart_features_raw.csv
Saved imputed version: ../data/processed/walmart_features_imputed.csv
